# 23 — When the table is too big: function approximation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/023-function-approximation.ipynb)

A Q-table has one row per state, so the table grows with every dimension you add to the state. In this notebook you count that growth, replace the table with four weights, and watch those four weights fit a curve over a continuous state. Then you break the model on purpose, twice.

## 1. The numbers: rows you would need

A grid with ten bins per dimension is already a lot of rows, and the count multiplies with each dimension. Read the log column: it is the number of zeros in the row count.

In [ ]:
import numpy as np                                   # the array library
bins = 10                                            # ten bins per dimension, the usual grid
dims = np.array([1, 2, 3, 6])                        # four ways of describing a state
for d in dims:                                       # one printed line per dimension count
    rows = bins ** d                                 # rows needed = 10 to the power d
    print(f"{d} dims -> {rows:>10,} rows, log10 {np.log10(rows):.1f}")   # count and its log
phi = np.array([1., 0.4, 0., 0.])                    # features for state 0.4, action 0
print("features:", phi)                              # always four numbers
print("stored:", phi.size, "against", bins ** 6)     # 4 against a million

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads the array library.</li><li><code>bins = 10</code> — how finely each dimension is cut.</li><li><code>dims = ...</code> — the four state descriptions we compare.</li><li><code>for d in dims:</code> — one line of output per dimension count.</li><li><code>rows = bins ** d</code> — the table has one row per cell of the grid.</li><li><code>print(f"...")</code> — the row count and its base-10 log.</li><li><code>phi = ...</code> — one feature row: bias, state, action, and their product.</li><li><code>print("features:", phi)</code> — the model's whole input, four numbers.</li><li><code>print("stored:", ...)</code> — four numbers replace a million-row table.</li></ul>

</details>

## 2. Step by step: fitting four weights

The model is `Q(s, a) = w0 + w1*s + w2*a + w3*s*a`, a straight-line fit in each action. We build the true values on a grid of states, then take 200 small steps downhill on the squared error. Watch the error and the weights in the printed checkpoints.

In [ ]:
states = np.linspace(0, 1, 11)                      # eleven states spread over the interval
actions = np.array([0., 1.])                         # two actions, written as numbers
X = []                                               # one feature row per (state, action)
for s in states:                                     # walk the states
    for a in actions:                                # walk the actions
        X.append([1., s, a, s * a])                  # bias, state, action, their product
X = np.array(X)                                      # shape (22, 4)
Y = []                                               # the true value of each row
for s in states:                                     # same states, same order
    for a in actions:                                # same actions, same order
        Y.append(s if a == 0 else 1. - s)            # action 0 pays s, action 1 pays 1 - s
Y = np.array(Y)                                      # shape (22,)
w = np.zeros(4)                                      # start with every weight at zero
lr = 0.3                                             # the step size
for step in range(200):                              # 200 sweeps over the same 22 rows
    pred = X @ w                                     # the current estimate for every row
    grad = X.T @ (pred - Y) / len(Y)                 # gradient of the mean squared error
    w = w - lr * grad                                # one step downhill
    if step % 50 == 0:                               # four checkpoints only
        mse = np.mean((X @ w - Y) ** 2)              # error right now
        print(f"step {step:3d}  mse {mse:.5f}  w {np.round(w, 2)}")   # watch it fall

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>states = np.linspace(0, 1, 11)</code> — eleven points cover the continuous state.</li><li><code>actions = ...</code> — two actions, coded as the numbers 0 and 1.</li><li><code>X = []</code> — will hold one row of features per state-action pair.</li><li><code>for s in states:</code> — outer loop over states.</li><li><code>for a in actions:</code> — inner loop over actions.</li><li><code>X.append([1., s, a, s * a])</code> — the four features; <code>s * a</code> lets each action bend.</li><li><code>X = np.array(X)</code> — 22 rows of four numbers.</li><li><code>Y = []</code> — the targets, built in exactly the same order.</li><li><code>for s in states:</code> — the same states again.</li><li><code>for a in actions:</code> — the same two actions again.</li><li><code>Y.append(s if a == 0 else 1. - s)</code> — the true value of each pair.</li><li><code>Y = np.array(Y)</code> — a matching column of 22 numbers.</li><li><code>w = np.zeros(4)</code> — the model starts knowing nothing.</li><li><code>lr = 0.3</code> — how far each step moves.</li><li><code>for step in range(200):</code> — the update loop.</li><li><code>pred = X @ w</code> — every prediction at once, as one matrix-vector product.</li><li><code>grad = X.T @ (pred - Y) / len(Y)</code> — slope of the mean squared error.</li><li><code>w = w - lr * grad</code> — move against the slope.</li><li><code>if step % 50 == 0:</code> — print only every fiftieth step.</li><li><code>mse = ...</code> — the error at this checkpoint.</li><li><code>print(...)</code> — error and weights together.</li></ul>

</details>

## 3. Plot: the fitted curve and the table's growth

The left panel puts the learned curve on top of the truth for both actions. The right panel returns to section 1 with both axes on a log scale, where a table's growth is a straight line and its slope is the number of dimensions.

In [ ]:
import matplotlib.pyplot as plt                      # plotting library
grid = np.linspace(0, 1, 101)                        # a fine grid for smooth curves
q0 = w[0] + w[1] * grid                              # Q(s, 0): bias and state weights only
q1 = q0 + w[2] + w[3] * grid                         # Q(s, 1): the action weights are added
fig, ax = plt.subplots(1, 2, figsize=(9, 3))         # two panels side by side
ax[0].plot(grid, grid, "k--", linewidth=1, label="true Q(s, 0)")      # truth, action 0
ax[0].plot(grid, 1 - grid, "r:", linewidth=1, label="true Q(s, 1)")   # truth, action 1
ax[0].plot(grid, q0, label="learned Q(s, 0)")        # fitted curve, action 0
ax[0].plot(grid, q1, label="learned Q(s, 1)")        # fitted curve, action 1
ax[0].set_xlabel("state s")                          # x-axis label
ax[0].set_ylabel("value Q(s, a)")                    # y-axis label
ax[0].legend(fontsize=7)                             # name the four lines
bins = np.array([2, 4, 8, 16, 32])                   # resolution per dimension
for d in [2, 3, 5]:                                  # three dimension counts
    ax[1].plot(bins, bins ** d, marker="o", label=f"{d} dims")   # rows = bins to the power d
ax[1].set_xscale("log")                              # log x
ax[1].set_yscale("log")                              # log y turns the exponent into a slope
ax[1].set_xlabel("bins per dimension (log)")         # x-axis label
ax[1].set_ylabel("table rows (log)")                 # y-axis label
ax[1].legend(fontsize=7)                             # name the three lines
fig.suptitle("four weights fit a curve; a table grows with every dimension")   # takeaway
plt.tight_layout()                                   # stop the labels overlapping
plt.show()                                           # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — loads the plotting library.</li><li><code>grid = ...</code> — 101 states for a smooth curve.</li><li><code>q0 = ...</code> — Q for action 0, using only two of the four weights.</li><li><code>q1 = ...</code> — Q for action 1; the other two weights shift and tilt it.</li><li><code>fig, ax = plt.subplots(1, 2, ...)</code> — one figure, two panels.</li><li><code>ax[0].plot(grid, grid, "k--", ...)</code> — the true action-0 value, a straight line.</li><li><code>ax[0].plot(grid, 1 - grid, ...)</code> — the true action-1 value.</li><li><code>ax[0].plot(grid, q0, ...)</code> — what the model learned for action 0.</li><li><code>ax[0].plot(grid, q1, ...)</code> — what it learned for action 1.</li><li><code>ax[0].set_xlabel(...)</code> — names the horizontal axis.</li><li><code>ax[0].set_ylabel(...)</code> — names the vertical axis.</li><li><code>ax[0].legend(fontsize=7)</code> — a key for the four lines.</li><li><code>bins = ...</code> — five grid resolutions.</li><li><code>for d in [2, 3, 5]:</code> — one line per dimension count.</li><li><code>ax[1].plot(bins, bins ** d, ...)</code> — rows against resolution.</li><li><code>ax[1].set_xscale("log")</code> — log x.</li><li><code>ax[1].set_yscale("log")</code> — log y; straight lines mean exponential growth.</li><li><code>ax[1].set_xlabel(...)</code> — names the horizontal axis.</li><li><code>ax[1].set_ylabel(...)</code> — names the vertical axis.</li><li><code>ax[1].legend(fontsize=7)</code> — a key for the three lines.</li><li><code>fig.suptitle(...)</code> — the sentence the whole figure is making.</li><li><code>plt.tight_layout()</code> — keeps the two panels from colliding.</li><li><code>plt.show()</code> — render.</li></ul>

</details>

## 4. Interactive: drag the step size

The slider sets `lr`, the distance travelled per step, and redraws the error curve on a log scale. Small steps crawl; large steps dive and then bounce around the answer. The title prints the error at the last sweep.

In [ ]:
from ipywidgets import interact                      # sliders that rerun a function
def fit(lr=0.3):                                     # lr comes from the slider
    ww = np.zeros(4)                                 # fresh weights on every drag
    hist = []                                        # error after each sweep
    for step in range(200):                          # the same 200 sweeps
        ww = ww - lr * X.T @ (X @ ww - Y) / len(Y)   # one gradient step
        hist.append(np.mean((X @ ww - Y) ** 2))      # record the error
    plt.figure(figsize=(5, 3))                       # a fresh figure per drag
    plt.plot(hist, color="tab:blue")                 # error against sweep number
    plt.yscale("log")                                # log y keeps the tail visible
    plt.xlabel("sweep")                              # x-axis label
    plt.ylabel("mean squared error (log)")           # y-axis label
    plt.title(f"lr = {lr:.2f}: final error {hist[-1]:.4f}")   # live readout
    plt.show()                                       # draw it
    print(f"lr {lr:.2f} -> final mse {hist[-1]:.5f}")   # the same number as text
interact(fit, lr=(0.01, 1.0, 0.01))                  # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the slider widget.</li><li><code>def fit(lr=0.3):</code> — one function per slider position.</li><li><code>ww = np.zeros(4)</code> — every drag starts from zero weights.</li><li><code>hist = []</code> — the error curve for this setting.</li><li><code>for step in range(200):</code> — the same 200 sweeps as section 2.</li><li><code>ww = ww - lr * ...</code> — one step, sized by the slider.</li><li><code>hist.append(...)</code> — store the error after the step.</li><li><code>plt.figure(...)</code> — a new figure for each drag.</li><li><code>plt.plot(hist, ...)</code> — the learning curve.</li><li><code>plt.yscale("log")</code> — the interesting part is the tail.</li><li><code>plt.xlabel("sweep")</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.title(...)</code> — the final error, printed on the plot.</li><li><code>plt.show()</code> — draw it.</li><li><code>print(...)</code> — the same number as text.</li><li><code>interact(fit, lr=(0.01, 1.0, 0.01))</code> — the slider, its range and its step.</li></ul>

</details>

## 5. When it breaks: two states, one pair of weights

Now the model sees a coarse bin instead of the exact state, so 0.1 and 0.2 arrive as the same row of features. Their targets are +1 and -1, and one shared weight cannot average those two numbers into anything that fits either. Alternate the updates and watch the two errors take turns being wrong.

In [ ]:
def features(s):                                     # a coarse-coded view of the state
    return np.array([1., np.floor(s / 0.5)])         # half-unit bins: 0.1 and 0.2 share bin 0
print("s=0.1 ->", features(0.1))                     # the row the model sees for 0.1
print("s=0.2 ->", features(0.2))                     # the identical row for 0.2
target = {0.1: 1.0, 0.2: -1.0}                       # opposite targets for the two states
w = np.zeros(2)                                      # one shared weight vector
lr = 0.5                                             # step size
e1, e2 = [], []                                      # error history at each state
for step in range(40):                               # 40 alternating updates
    s = 0.1 if step % 2 == 0 else 0.2                # which state we train on this step
    phi = features(s)                                # identical features either way
    err = phi @ w - target[s]                        # signed error on this state
    w = w - lr * err * phi                           # one least-mean-squares step
    e1.append(abs(features(0.1) @ w - 1.0))          # damage left at state 0.1
    e2.append(abs(features(0.2) @ w + 1.0))          # damage left at state 0.2
plt.figure(figsize=(6, 3))                           # new figure
plt.plot(e1, label="error at s = 0.1 (target +1)")   # first track
plt.plot(e2, label="error at s = 0.2 (target -1)")   # second track
plt.ylim(0, 2.2)                                     # both errors on one scale
plt.xlabel("update number")                          # x-axis label
plt.ylabel("absolute error")                         # y-axis label
plt.legend(fontsize=8)                               # name the two tracks
plt.title("fixing one state breaks the other, forever")   # takeaway
plt.show()                                           # render
print(f"last errors: {e1[-1]:.2f} and {e2[-1]:.2f}")  # neither ever reaches zero

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>def features(s):</code> — the model's whole view of a state.</li><li><code>return np.array([1., np.floor(s / 0.5)])</code> — bin number instead of the exact state.</li><li><code>print("s=0.1 -&gt;", ...)</code> — the first state's features.</li><li><code>print("s=0.2 -&gt;", ...)</code> — identical, so nothing can separate them.</li><li><code>target = {...}</code> — one state wants +1, the other wants -1.</li><li><code>w = np.zeros(2)</code> — a single shared weight vector.</li><li><code>lr = 0.5</code> — step size.</li><li><code>e1, e2 = [], []</code> — two error histories.</li><li><code>for step in range(40):</code> — 40 updates.</li><li><code>s = 0.1 if step % 2 == 0 else 0.2</code> — alternate the state.</li><li><code>phi = features(s)</code> — its features.</li><li><code>err = phi @ w - target[s]</code> — prediction minus target.</li><li><code>w = w - lr * err * phi</code> — move the shared weights toward this state.</li><li><code>e1.append(...)</code> — how far state 0.1 now is from +1.</li><li><code>e2.append(...)</code> — how far state 0.2 now is from -1.</li><li><code>plt.figure(...)</code> — a new figure.</li><li><code>plt.plot(e1, ...)</code> — the first error track.</li><li><code>plt.plot(e2, ...)</code> — the second.</li><li><code>plt.ylim(0, 2.2)</code> — both tracks on one readable scale.</li><li><code>plt.xlabel("update number")</code> — names the horizontal axis.</li><li><code>plt.ylabel("absolute error")</code> — names the vertical axis.</li><li><code>plt.legend(fontsize=8)</code> — name the two tracks.</li><li><code>plt.title(...)</code> — the sentence the figure is making.</li><li><code>plt.show()</code> — render.</li><li><code>print(f"last errors ...")</code> — the residual error at the end.</li></ul>

</details>

## Your turn

Change the bin width in `features` to 0.05 and predict what happens to the two error tracks before you rerun. Then set `lr` in section 4 to 1.5 and describe the curve you get.